# Phase 2a — PySpark Processing & Data Cleaning

## Why PySpark instead of pandas here?

Pandas loads the **entire dataset into RAM** on a single CPU core. For our 49k-row dataset that's fine today — but a real data engineering pipeline must handle data that grows. PySpark:

- Distributes work across **multiple cores** (we use 4 here with `local[4]`)
- Can scale to a **cluster** without rewriting a single line of code
- Reads and writes **Parquet** natively — a columnar format that is 5-10x smaller and faster than CSV
- Has built-in **schema enforcement** — it will throw an error if data types don't match, catching bugs early

In production, you'd submit this as a `spark-submit` job. In development, we run it locally.

## Windows Setup (IMPORTANT — read before running)

PySpark on Windows requires:
1. **Java 8 or 11** — NOT Java 17+. Download from https://adoptium.net/
2. **winutils.exe** — Hadoop compatibility shim for Windows:
   - Download: https://github.com/cdarlint/winutils (pick the hadoop-3.3.x folder)
   - Place `winutils.exe` in `C:\hadoop\bin\`
   - Set environment variable: `HADOOP_HOME = C:\hadoop`
3. **JAVA_HOME** — must point to your JDK root folder

The cell below sets these programmatically. **Update the path to match your Java install.**

In [ ]:
# ============================================================
# Cell 2 — Initialize SparkSession
# ============================================================
import os
import sys
from pathlib import Path

# ---- Windows: Set JAVA_HOME and HADOOP_HOME BEFORE importing pyspark ----
# These must be set before pyspark starts the JVM.
# Find your Java install path by running: where java  (in cmd)
# Then go one level up from the bin/ folder.

# Auto-detect common Java install locations on Windows
_java_candidates = [
    r"C:\Program Files\Eclipse Adoptium\jdk-11.0.23.9-hotspot",
    r"C:\Program Files\Eclipse Adoptium\jdk-11.0.22.9-hotspot",
    r"C:\Program Files\Microsoft\jdk-11.0.22.7-hotspot",
    r"C:\Program Files\Java\jdk-11",
    r"C:\Program Files\Java\jdk1.8.0_401",
    r"C:\Program Files\OpenJDK\jdk-11",
]

if "JAVA_HOME" not in os.environ:
    for candidate in _java_candidates:
        if os.path.exists(candidate):
            os.environ["JAVA_HOME"] = candidate
            print(f"Auto-detected JAVA_HOME: {candidate}")
            break
    else:
        print("WARNING: JAVA_HOME not found automatically.")
        print("Set it manually: os.environ['JAVA_HOME'] = r'C:\\path\\to\\jdk'")
else:
    print(f"JAVA_HOME already set: {os.environ['JAVA_HOME']}")

# Set Hadoop home for winutils.exe (Windows only)
if sys.platform == "win32" and "HADOOP_HOME" not in os.environ:
    os.environ["HADOOP_HOME"] = r"C:\hadoop"
    print(f"Set HADOOP_HOME: {os.environ['HADOOP_HOME']}")
    print("Make sure winutils.exe exists at C:\\hadoop\\bin\\winutils.exe")

# ---- Import PySpark ----
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ---- Create SparkSession ----
# local[4] = run on this machine using 4 CPU cores
# In production this would be: spark://master:7077 or yarn/kubernetes
spark = (
    SparkSession.builder
    .appName("FIFA_WC2026_Processing")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")   # Reduce from default 200 for small data
    .config("spark.driver.memory", "2g")            # Give the driver 2GB RAM
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")  # Handle date edge cases
    .getOrCreate()
)

# Suppress noisy INFO logs — only show WARN and above
spark.sparkContext.setLogLevel("WARN")

print(f"\nSpark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"Master        : {spark.sparkContext.master}")
print("\nSparkSession ready.")

## Reading Raw CSVs into Spark DataFrames

Spark can infer schemas automatically (`inferSchema=True`), but we define them **explicitly** for two reasons:
1. Inference requires a full scan of the file — slow on large datasets
2. Explicit schemas catch data type mismatches at read time, not silently later

We also tell Spark the CSV has a **header row** — otherwise it treats column names as data.

In [ ]:
# ============================================================
# Cell 3 — Read raw CSVs into Spark DataFrames
# ============================================================

# Resolve path to data/raw/ from the notebooks/ folder
RAW_DATA = str(Path("..") / "data" / "raw")

# ---- Define explicit schemas (better than inferSchema for production) ----
results_schema = T.StructType([
    T.StructField("date",        T.StringType(),  True),   # Will parse to DateType below
    T.StructField("home_team",   T.StringType(),  False),
    T.StructField("away_team",   T.StringType(),  False),
    T.StructField("home_score",  T.IntegerType(), True),
    T.StructField("away_score",  T.IntegerType(), True),
    T.StructField("tournament",  T.StringType(),  True),
    T.StructField("city",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("neutral",     T.StringType(),  True),   # 'TRUE'/'FALSE' string
])

goalscorers_schema = T.StructType([
    T.StructField("date",       T.StringType(),  True),
    T.StructField("home_team",  T.StringType(),  True),
    T.StructField("away_team",  T.StringType(),  True),
    T.StructField("team",       T.StringType(),  True),
    T.StructField("scorer",     T.StringType(),  True),
    T.StructField("minute",     T.DoubleType(),  True),    # Double because some have .5 (added time)
    T.StructField("own_goal",   T.StringType(),  True),
    T.StructField("penalty",    T.StringType(),  True),
])

shootouts_schema = T.StructType([
    T.StructField("date",          T.StringType(), True),
    T.StructField("home_team",     T.StringType(), True),
    T.StructField("away_team",     T.StringType(), True),
    T.StructField("winner",        T.StringType(), True),
    T.StructField("first_shooter", T.StringType(), True),
])

former_names_schema = T.StructType([
    T.StructField("current",    T.StringType(), True),
    T.StructField("former",     T.StringType(), True),
    T.StructField("start_date", T.StringType(), True),
    T.StructField("end_date",   T.StringType(), True),
])

# ---- Read CSVs ----
df_results      = spark.read.csv(f"{RAW_DATA}/results.csv",      schema=results_schema,      header=True)
df_goalscorers  = spark.read.csv(f"{RAW_DATA}/goalscorers.csv",  schema=goalscorers_schema,  header=True)
df_shootouts    = spark.read.csv(f"{RAW_DATA}/shootouts.csv",    schema=shootouts_schema,    header=True)
df_former_names = spark.read.csv(f"{RAW_DATA}/former_names.csv", schema=former_names_schema, header=True)

# ---- Print counts to confirm read ----
dfs = {
    "results":      df_results,
    "goalscorers":  df_goalscorers,
    "shootouts":    df_shootouts,
    "former_names": df_former_names,
}

print(f"{'Dataset':<15} {'Rows':>8} {'Partitions':>12}")
print("-" * 38)
for name, df in dfs.items():
    print(f"{name:<15} {df.count():>8,} {df.rdd.getNumPartitions():>12}")

print("\nResults schema:")
df_results.printSchema()

## Cleaning & Standardization

Before any analysis or feature engineering, we need clean, consistent data. Issues we fix here:

| Problem | Fix |
|---------|-----|
| Dates stored as strings | Cast to `DateType` |
| 'TRUE'/'FALSE' strings for boolean columns | Cast to `BooleanType` |
| Historical team names (e.g. "West Germany") | Map to current name using `former_names.csv` |
| Null scores (some very old matches have null) | Drop those rows — we can't use them |
| Negative or implausible scores | Flag and drop |

**Why normalize team names?**  
If we don't, "West Germany" and "Germany" are treated as two separate teams. Their combined match history is split, making features for Germany look artificially sparse and recent.

In [ ]:
# ============================================================
# Cell 4 — Clean and standardize
# ============================================================

# ---- Step 1: Build the name normalization map ----
# former_names.csv maps old names to current names.
# We broadcast this small dataframe so every Spark worker has a local copy
# instead of shuffling it across the network on every join.

# Collect former->current mapping as Python dict (safe — small dataset)
name_map = {
    row["former"]: row["current"]
    for row in df_former_names.collect()
    if row["former"] is not None and row["current"] is not None
}
print(f"Name normalization map: {len(name_map)} entries")
print("Sample:", dict(list(name_map.items())[:5]))

# Convert Python dict to a Spark DataFrame for joining
# We need TWO joins: one for home_team, one for away_team
name_map_rows = [(former, current) for former, current in name_map.items()]
df_name_map = spark.createDataFrame(name_map_rows, ["former_name", "current_name"])

# ---- Step 2: Clean the results table ----
df_clean = (
    df_results
    # Parse date string 'YYYY-MM-DD' → DateType
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    # Cast boolean columns (stored as 'TRUE'/'FALSE' strings)
    .withColumn("neutral", F.col("neutral").cast(T.BooleanType()))
    # Drop rows with null scores — can't engineer features without them
    .dropna(subset=["home_score", "away_score", "home_team", "away_team", "date"])
    # Drop rows with implausible negative scores (data errors)
    .filter((F.col("home_score") >= 0) & (F.col("away_score") >= 0))
    # Add result column: W=home win, D=draw, L=home loss
    .withColumn(
        "result",
        F.when(F.col("home_score") > F.col("away_score"), F.lit("W"))
         .when(F.col("home_score") == F.col("away_score"), F.lit("D"))
         .otherwise(F.lit("L"))
    )
    # Add goal_diff from home team perspective (used in feature engineering)
    .withColumn("goal_diff", F.col("home_score") - F.col("away_score"))
)

# ---- Step 3: Normalize team names via left join ----
# Join home_team against the mapping; if no match, keep original name
df_clean = (
    df_clean
    .join(
        df_name_map.withColumnRenamed("former_name", "home_team")
                   .withColumnRenamed("current_name", "home_team_norm"),
        on="home_team",
        how="left"
    )
    .withColumn(
        "home_team",
        F.coalesce(F.col("home_team_norm"), F.col("home_team"))  # Use normalized if available
    )
    .drop("home_team_norm")
    # Same for away_team
    .join(
        df_name_map.withColumnRenamed("former_name", "away_team")
                   .withColumnRenamed("current_name", "away_team_norm"),
        on="away_team",
        how="left"
    )
    .withColumn(
        "away_team",
        F.coalesce(F.col("away_team_norm"), F.col("away_team"))
    )
    .drop("away_team_norm")
    # Strip leading/trailing whitespace from team names (common data issue)
    .withColumn("home_team", F.trim(F.col("home_team")))
    .withColumn("away_team", F.trim(F.col("away_team")))
    # Sort by date for downstream window functions
    .orderBy("date")
)

# ---- Step 4: Clean goalscorers ----
df_goals_clean = (
    df_goalscorers
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("own_goal", F.col("own_goal").cast(T.BooleanType()))
    .withColumn("penalty",  F.col("penalty").cast(T.BooleanType()))
    .withColumn("minute",   F.col("minute").cast(T.IntegerType()))
    .withColumn("team",     F.trim(F.col("team")))
    .dropna(subset=["date", "team"])
)

# ---- Step 5: Clean shootouts ----
df_shootouts_clean = (
    df_shootouts
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("home_team", F.trim(F.col("home_team")))
    .withColumn("away_team", F.trim(F.col("away_team")))
    .withColumn("winner",    F.trim(F.col("winner")))
    .dropna(subset=["date", "winner"])
)

# ---- Print cleaning report ----
original_count = df_results.count()
clean_count    = df_clean.count()
dropped        = original_count - clean_count

print(f"\nResults cleaning report:")
print(f"  Original rows : {original_count:,}")
print(f"  Clean rows    : {clean_count:,}")
print(f"  Rows dropped  : {dropped:,} ({dropped/original_count*100:.2f}%)")

print("\nResult distribution after cleaning:")
df_clean.groupBy("result").count().orderBy("result").show()

print("\nSample cleaned results:")
df_clean.select("date", "home_team", "away_team", "home_score", "away_score", "result", "goal_diff").show(5)

## Writing to Parquet

**Why Parquet instead of CSV?**

| Feature | CSV | Parquet |
|---------|-----|----------|
| Storage | ~10MB (results) | ~1-2MB (compressed) |
| Read speed | Slow (parse every row) | Fast (columnar, skip columns) |
| Schema | None (strings only) | Embedded (types preserved) |
| Compression | None | Snappy (default) |
| Supported by | Everything | Spark, dbt, pandas, Arrow |

We write as **overwrite** mode — re-running this notebook replaces the files safely.

In [ ]:
# ============================================================
# Cell 5 — Write cleaned data to Parquet
# ============================================================

PROCESSED_DATA = str(Path("..") / "data" / "processed")

# ---- Write each cleaned dataset as Parquet ----
# coalesce(1) writes a single .parquet file instead of many small ones
# For small datasets this is fine; for large data use repartition() instead

write_jobs = [
    (df_clean,            f"{PROCESSED_DATA}/matches_clean.parquet"),
    (df_goals_clean,      f"{PROCESSED_DATA}/goalscorers_clean.parquet"),
    (df_shootouts_clean,  f"{PROCESSED_DATA}/shootouts_clean.parquet"),
]

for df, path in write_jobs:
    (
        df.coalesce(1)       # Single file output for small dataset
          .write
          .mode("overwrite") # Safe to re-run
          .parquet(path)
    )
    print(f"Written: {path}")

print("\nAll parquet files written successfully.")

## Verification

We read the Parquet files back to confirm:
1. Row counts match what we wrote
2. Schemas are correctly preserved (dates as `DateType`, not strings)
3. The data looks correct after team name normalization

In [ ]:
# ============================================================
# Cell 6 — Verify parquet output
# ============================================================

print("Reading parquet files back for verification...\n")

df_matches_v    = spark.read.parquet(f"{PROCESSED_DATA}/matches_clean.parquet")
df_goals_v      = spark.read.parquet(f"{PROCESSED_DATA}/goalscorers_clean.parquet")
df_shootouts_v  = spark.read.parquet(f"{PROCESSED_DATA}/shootouts_clean.parquet")

# ---- Schema check ----
print("matches_clean schema (confirm date is DateType, not StringType):")
df_matches_v.printSchema()

# ---- Row count check ----
print(f"{'File':<30} {'Rows':>8}")
print("-" * 40)
print(f"{'matches_clean.parquet':<30} {df_matches_v.count():>8,}")
print(f"{'goalscorers_clean.parquet':<30} {df_goals_v.count():>8,}")
print(f"{'shootouts_clean.parquet':<30} {df_shootouts_v.count():>8,}")

# ---- Spot-check name normalization ----
# 'West Germany' should have been replaced with 'Germany'
print("\nConfirm 'West Germany' no longer appears (should show 0 rows):")
df_matches_v.filter(
    (F.col("home_team") == "West Germany") | (F.col("away_team") == "West Germany")
).count()
count_wg = df_matches_v.filter(
    (F.col("home_team") == "West Germany") | (F.col("away_team") == "West Germany")
).count()
print(f"  Rows with 'West Germany': {count_wg}  ({'PASS' if count_wg == 0 else 'FAIL — check name mapping'})")

# ---- Date range ----
date_stats = df_matches_v.agg(
    F.min("date").alias("earliest"),
    F.max("date").alias("latest")
).collect()[0]
print(f"\n  Date range: {date_stats['earliest']} → {date_stats['latest']}")

# ---- Sample rows ----
print("\nSample rows from matches_clean.parquet:")
df_matches_v.select(
    "date", "home_team", "away_team",
    "home_score", "away_score", "result", "tournament"
).orderBy(F.col("date").desc()).show(5)

# Stop Spark to free resources before moving to next notebook
spark.stop()
print("SparkSession stopped.")